# API da busca de orientadores (Colab + GPU)

Roda o **pré-ranking** (busca híbrida no Qdrant) e o **reranking** (bge-reranker-v2-m3) na GPU do Colab
e expõe uma **API JSON com link público** para o seu front-end exibir o resultado.

**Antes de rodar:**
1. `Ambiente de execução > Alterar o tipo de ambiente de execução > GPU (T4)`.
2. Na barra lateral, em 🔑 **Secrets**, crie (e libere o acesso deste notebook):
   - `QDRANT_URL` e `QDRANT_API_KEY` (os mesmos do `.env`);
   - `GITHUB_TOKEN` **só se o repositório for privado** (token com permissão de leitura).
3. Confira `BRANCH` na célula de clone: o Colab usa o código **que está no GitHub** (commit + push antes).

Depois é só `Ambiente de execução > Executar tudo`. A última célula de setup imprime o link da API.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Sem GPU: ative em Ambiente de execução > Alterar o tipo"

## 1. Código e dependências

In [ ]:
import os
from google.colab import userdata

REPO = "gavgms12/tcc_repository"
BRANCH = "buscas"
PASTA = "/content/tcc_repository"

try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

if os.path.exists(PASTA):
    !git -C {PASTA} fetch -q origin {BRANCH} && git -C {PASTA} checkout -q {BRANCH} && git -C {PASTA} reset -q --hard origin/{BRANCH}
else:
    !git clone -q --branch {BRANCH} {url} {PASTA}
!git -C {PASTA} log -1 --oneline

In [ ]:
# O torch com CUDA já vem no Colab; aqui entram só as libs da busca e da API.
!pip install -q -r {PASTA}/requirements-busca.txt fastapi uvicorn pandas python-dotenv
# Se o pip pedir para reiniciar a sessão: reinicie e rode a partir da célula de clone (é rápido na 2ª vez).

## 2. Modelos e API

In [ ]:
import sys, torch

os.environ["QDRANT_URL"] = userdata.get("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = userdata.get("QDRANT_API_KEY")
os.chdir(PASTA)
if PASTA not in sys.path:
    sys.path.insert(0, PASTA)

print("GPU:", torch.cuda.is_available())

from recuperacao.sistema_de_busca.api import carregar_app
app = carregar_app()   # baixa/carrega bge-m3 e bge-reranker-v2-m3 (1ª vez demora alguns minutos)

In [ ]:
# Sobe a API em segundo plano (porta 8000) e aquece os modelos com uma busca de teste.
import threading, time, requests, uvicorn

servidor = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=servidor.run, daemon=True).start()
while not servidor.started:
    time.sleep(0.2)

r = requests.post("http://localhost:8000/recomendar", json={"texto": "algoritmos em grafos", "top_k": 3})
print(r.status_code, [p["nome"] for p in r.json()["ranking"]], r.json()["tempos"])

In [ ]:
# Link público (Cloudflare Quick Tunnel: sem conta). O link muda a cada vez que esta célula roda.
import re, subprocess

if not os.path.exists("/content/cloudflared"):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /content/cloudflared

tunel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL_API = None
for linha in tunel.stdout:
    achado = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", linha)
    if achado:
        URL_API = achado.group(0)
        break
# continua lendo o log em segundo plano (se o buffer encher, o túnel trava)
threading.Thread(target=lambda: [None for _ in tunel.stdout], daemon=True).start()
print("API:", URL_API)
print("Documentação interativa:", f"{URL_API}/docs")

## 3. Usando a API

`POST {URL_API}/recomendar` com JSON:

| campo | tipo | padrão | |
|---|---|---|---|
| `texto` | string | — | texto livre do aluno |
| `topicos` | lista | — | tópicos do catálogo (`GET /topicos`); pode combinar com `texto` |
| `top_k` | int | 5 | professores devolvidos |
| `n_itens` | int | 3 | evidências por tipo em `destaques` |
| `ano_min`, `ano_max` | int | — | filtro de ano dos itens |
| `incluir_pre_ranking` | bool | false | devolve também o ranking antes do reranking |

Resposta: `ranking` (lista de professores com `nome`, `score`, `score_inicial`, `scores_topicos`,
`perfil` = nota do texto de perfil e `parcela_do_score` (fração do score que veio dele),
`destaques` = melhores itens por tipo, e `topicos` = detalhe por tópico), `professores_cortados`, `tempos`
e, se pedido, `pre_ranking`.

Texto livre: frases de intenção ("quero pesquisar", "tenho interesse em"...) são removidas e o texto é
quebrado em ideias (`,` `;` `.` " e " " ou ") para o reranker; cada item fica com a maior nota entre elas.

Exemplo no front-end (JavaScript):
```js
const resp = await fetch(`${URL_API}/recomendar`, {
  method: "POST",
  headers: {"Content-Type": "application/json"},
  body: JSON.stringify({texto: "quero pesquisar sistemas embarcados e IoT", top_k: 5}),
});
const {ranking} = await resp.json();
```

**Limites do Colab:** a sessão cai após ~90 min sem interação (ou ~12 h no total) e o link muda a cada execução.
Serve para testes e demonstração; para deixar no ar de forma fixa é preciso hospedar a API em um servidor.

In [ ]:
# Teste pelo link público (com fallback para localhost se o DNS ainda não propagou)
import json
pedido = {"texto": "quero pesquisar sistemas embarcados e internet das coisas", "top_k": 5}
try:
    r = requests.post(f"{URL_API}/recomendar", json=pedido)
except requests.exceptions.ConnectionError:
    print("⚠️ O link público ainda está propagando no DNS. Testando localmente via localhost...")
    r = requests.post("http://localhost:8000/recomendar", json=pedido)

for prof in r.json()["ranking"]:
    perfil = prof["perfil"]
    parcela = f", {perfil['parcela_do_score']:.0%} do score" if perfil["parcela_do_score"] is not None else ""
    print(f"{prof['posicao']}º {prof['nome']}  (score {prof['score']:.4f})")
    print(f"    perfil: {perfil['score']:.3f}{parcela}")
    for tipo, itens in prof["destaques"].items():
        for item in itens:
            codigo = f"[{item['codigo']}] " if item.get("codigo") else ""
            print(f"    {tipo}: {codigo}{item['titulo']}  ({item['score']:.3f})")
print("tempos:", json.dumps(r.json()["tempos"], indent=1))